# CUSTOMER

## Create the gold layer

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace2.gold

## Create the dimention customer query

In [0]:

query = """
    SELECT 
        ROW_NUMBER() OVER(ORDER BY customer_id) AS customer_number,
        c.customer_id,
        c.customer_key,
        c.firstname,
        c.lastname,
        c.marital_status,
        CASE
            WHEN c.gender != 'n/a' THEN c.gender
            ELSE COALESCE(cs.gender, 'n/a')
        END AS gender,
        c.creation_date,
        cs.birth_date,
        l.country
    FROM workspace2.silver.customer AS c
    LEFT JOIN workspace2.silver.cust_az AS cs
    ON c.customer_key = cs.customer_key
    LEFT JOIN workspace2.silver.loc_a1 AS l
    ON c.customer_key = l.customer_key
"""
 

## Ingest dim_customer gold layer

In [0]:
dim_customer = spark.sql(query)
dim_customer.write.mode("overwrite").saveAsTable("workspace2.gold.dim_customers")


In [0]:
display(dim_customer.count())

## Create the dimention product query

In [0]:
query = """
    SELECT 
        ROW_NUMBER() OVER(ORDER BY product_id) AS product_number,
        p.product_id,
        p.product_key,
        p.category_id,
        p.product_name,
        p.product_cost,
        p.product_line,
        p.product_start_date,
        p.product_end_date,
        pc.CAT AS category,
        pc.SUBCAT AS sub_category,
        pc.MAINTENANCE AS maintenance
    FROM workspace2.silver.products p
    LEFT JOIN workspace2.silver.px_cat pc
    ON p.category_id = pc.ID
        """

## Ingest dim_products gold layer

In [0]:
dim_products = spark.sql(query)
dim_products.write.mode("overwrite").saveAsTable("workspace2.gold.dim_products")

dim_products.display()

## Create the fact sales query

In [0]:
query2 = """
    SELECT
        s.order_id,
        c.customer_number,
        p.product_number,
        s.order_date,
        s.ship_date,
        s.due_date,
        s.sales,
        s.quantity,
        s.price
    FROM workspace2.silver.sales s
    LEFT JOIN workspace2.gold.dim_customers c
    ON s.customer_id = c.customer_id
    LEFT JOIN workspace2.gold.dim_products p
    ON s.product_key = p.product_key

"""

## Ingest fact sales gold layer

In [0]:
fact_sales = spark.sql(query2)
fact_sales.write.mode("overwrite").saveAsTable("workspace2.gold.fact_sales")

fact_sales.display()